# Session 28: Imbalanced Data

Data Science with Python, Level 2, Stage 4: Applied ML

## Learning Objectives

By the end of this session you will be able to:

1. Recognise an imbalanced classification problem.
2. Explain why accuracy fails on rare classes.
3. Use stratified splits and folds.
4. Apply class weights.
5. Oversample and undersample the training data.
6. Use precision-recall curves and thresholds.

## Where We Are

In the 6,000 practice card payments, only **108** are fraud: 1.8 percent. Many important problems look like this: machine failures, rare diseases, loan defaults. The interesting class is rare, and a model can score 98 percent accuracy by never predicting it.

Today you learn the standard toolkit for **imbalanced data**: stratified splits, class weights, resampling and threshold choice, with precision and recall from Session 21 as the scoreboard. We start with a simple version of the bank's system that stores only the **amount**, the **hour** and whether the payment was **online**.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score,
                             precision_recall_curve, average_precision_score, roc_auc_score)

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
tx = pd.read_csv(DATA_URL + "card_transactions.csv")
cols = ["amount", "hour", "online"]
X = tx[cols].astype(float)
y = tx["is_fraud"]

## 1. The Accuracy Trap

In [ ]:
print(y.value_counts())
print("Fraud rate:", round(y.mean(), 3))

In [ ]:
for seed in range(5):
    _, _, _, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)
    print("seed", seed, "frauds in test set:", y_test.sum())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=y)

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print(confusion_matrix(y_test, pred))

## 2. Class Weights

`class_weight="balanced"` tells the model that each fraud counts as much as about 55 normal payments during training, the ratio of the two classes. The model then pays far more attention to the rare class. Most scikit-learn classifiers accept it.

In [ ]:
for weights in [None, "balanced"]:
    m = make_pipeline(StandardScaler(), LogisticRegression(class_weight=weights))
    p = m.fit(X_train, y_train).predict(X_test)
    print(weights, "recall", round(recall_score(y_test, p), 2),
          "precision", round(precision_score(y_test, p), 2))

## 3. Resampling

In [ ]:
train = X_train.assign(is_fraud=y_train)
fraud = train[train["is_fraud"] == 1]
normal = train[train["is_fraud"] == 0]

copies = fraud.sample(len(normal), replace=True, random_state=0)
oversampled = pd.concat([normal, copies])
undersampled = pd.concat([normal.sample(5 * len(fraud), random_state=0), fraud])
print(len(oversampled), len(undersampled))

In [ ]:
for name, data in [("over", oversampled), ("under", undersampled)]:
    m = make_pipeline(StandardScaler(), LogisticRegression())
    p = m.fit(data[cols], data["is_fraud"]).predict(X_test)
    print(name, "recall", round(recall_score(y_test, p), 2),
          "precision", round(precision_score(y_test, p), 2))

| Tool | How | Watch out |
| --- | --- | --- |
| stratify / StratifiedKFold | Keep the class ratio in every split | Always use it |
| class_weight="balanced" | Rare class errors cost more | Simplest first step |
| Oversampling | Copy rare rows (or SMOTE: synthetic rows) | Only on training data |
| Undersampling | Drop common rows | Throws away information |
| Threshold | Move the cut-off from 0.5 | Choose on validation data |

SMOTE comes from the imbalanced-learn package: pip install imbalanced-learn.

## 4. Thresholds and the PR Curve

In [ ]:
proba = model.predict_proba(X_test)[:, 1]
for t in [0.5, 0.2, 0.1, 0.05, 0.02]:
    p = (proba >= t).astype(int)
    print(t, "recall", round(recall_score(y_test, p), 2),
          "precision", round(precision_score(y_test, p), 2))

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, proba)
ap = average_precision_score(y_test, proba)

plt.plot(recall, precision)
plt.axhline(y_test.mean(), color="gray", linestyle="--", label="Random")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-recall curve, AP = {ap:.2f}")
plt.legend()
plt.show()

In [ ]:
for scoring in ["roc_auc", "average_precision"]:
    s = cross_val_score(model, X, y, scoring=scoring,
                        cv=StratifiedKFold(5, shuffle=True, random_state=0))
    print(scoring, round(s.mean(), 3))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Resampling the test set | Oversampling before splitting | Split first, resample only the training data. |
| Accuracy again | Reporting 98.8 percent accuracy | Report recall, precision and average precision. |
| Forgetting stratify | Random splits with very few positives | Use stratified splits and folds. |
| Threshold on test | Choosing the threshold that looks best on the test set | Choose it with cross-validation or a validation set. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Weights for Boosting

Train `HistGradientBoostingClassifier(learning_rate=0.05, max_depth=3)` with and without `class_weight="balanced"`. Print the confusion matrix and average precision of each.

In [ ]:
# Write your code below

### In-Class Exercise 2: Cost of Mistakes

A missed fraud costs 270 and a false alarm costs 5. For thresholds 0.5, 0.2, 0.1, 0.05 and 0.02 on the plain logistic model, compute the total cost on the test set. Which threshold is cheapest?

In [ ]:
# Write your code below

### In-Class Exercise 3: Custom Weights

Instead of `balanced`, try `class_weight={0: 1, 1: w}` for w in 1, 5, 20 and 55. Print recall and precision for each. How does the weight move the balance?

In [ ]:
# Write your code below

### In-Class Exercise 4: All the Features

Add `distance_from_home_km` and `new_merchant` to the features. Report the cross-validated average precision. What does the result tell you about which information matters most?

In [ ]:
# Write your code below

## Summary

- With rare classes, accuracy is misleading.
- Always stratify splits and cross-validation folds.
- `class_weight="balanced"` makes rare class errors cost more.
- Over- and undersampling change the training data only.
- Thresholds and weights are two routes to the same trade-off.
- Use precision, recall and average precision as the scoreboard.

## What's Next

You have now used imputers, encoders, scalers and models. Next session you will put every step, including custom ones you write yourself, into one pipeline that can be cross-validated, saved and used in an app.